In [2]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression

In [9]:
df = []

df_1264 = pd.read_csv('1264_nau_courses.csv')
df_1261 = pd.read_csv('1261_nau_courses.csv')

df.append(df_1264)
df.append(df_1261)

df = pd.concat(df)

df.to_csv('nau_courses_ai.csv', index=False)
df

,subject,course_id,course_display,description,course_url
0,ACC,4,ACC 199 - Special Topics,Foundations of intellectual inquiry. In-depth ...,https://catalog.nau.edu/Courses/course?courseI...
1,ACC,799,ACC 205 - Introduction To Business Law,"An introduction to business-related legal, reg...",https://catalog.nau.edu/Courses/course?courseI...
2,ACC,10175,ACC 205H - Introduction To Business Law - Honors,"An introduction to business-related legal, reg...",https://catalog.nau.edu/Courses/course?courseI...
3,ACC,11574,ACC 206 - Language For Business Law,This is language- and content-enrichment cours...,https://catalog.nau.edu/Courses/course?courseI...
4,ACC,5,ACC 255 - Financial Accounting For Business,This course introduces the basic financial acc...,https://catalog.nau.edu/Courses/course?courseI...
...,...,...,...,...,...
6157,TH,12214,TH 490CH - Senior Capstone Experience - Honors,Seminar for seniors including career workshops...,https://catalog.nau.edu/Courses/course?courseI...
6158,TH,8352,TH 497 - Independent Study,Individualized approach to selected topics by ...,https://catalog.nau.edu/Courses/course?courseI...
6159,TH,8353,TH 499 - Contemporary Developments,Examines recent trends and investigations in a...,https://catalog.nau.edu/Courses/course?courseI...
6160,TH,8359,TH 599 - Contemporary Developments,Examines recent trends and investigations in a...,https://catalog.nau.edu/Courses/course?courseI...


In [ ]:
import argparse
import os
import re

import pandas as pd


TERM_RE = re.compile(r"(\d{4})")


def extract_term(path: str) -> str:
    base = os.path.basename(path)
    match = TERM_RE.search(base)
    return match.group(1) if match else "unknown"


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--in", dest="inputs", nargs="+", required=True, help="Input CSV files")

    ap.add_argument("--out", default="nau_courses_combined.csv", help="Output CSV path")
    ap.add_argument(
        "--dedup-out",
        default="nau_courses_dedup.csv",
        help="Output CSV path after removing duplicates",
    )
    ap.add_argument(
        "--diff-out",
        default="nau_courses_description_changes.csv",
        help="CSV for courses whose descriptions changed across terms",
    )
    args = ap.parse_args()

    frames = []
    args.inputs = ['1264_nau_courses.csv', '1261_nau_courses.csv']
    for path in args.inputs:
        df = pd.read_csv(path)
        df["term"] = extract_term(path)
        frames.append(df)

    combined = pd.concat(frames, ignore_index=True)
    combined.to_csv(args.out, index=False)
    print(f"Saved combined file: {args.out} ({len(combined):,} rows)")

    # Normalize description for comparison across terms.
    combined["description_norm"] = (
        combined["description"].fillna("").str.strip().str.replace(r"\s+", " ", regex=True)
    )

    # Identify course_ids with multiple distinct descriptions across terms.
    desc_counts = (
        combined.groupby("course_id")["description_norm"].nunique(dropna=False).reset_index()
    )
    changed_ids = desc_counts[desc_counts["description_norm"] > 1]["course_id"]
    changes = combined[combined["course_id"].isin(changed_ids)].copy()
    changes.to_csv(args.diff_out, index=False)
    print(f"Saved description changes: {args.diff_out} ({len(changes):,} rows)")

    # Remove duplicates across terms (same course_id) after confirming description matches.
    dedup = combined[~combined["course_id"].isin(changed_ids)].copy()
    dedup = dedup.drop(columns=["description_norm"]).drop_duplicates(subset=["course_id"])
    dedup.to_csv(args.dedup_out, index=False)
    print(f"Saved deduped file: {args.dedup_out} ({len(dedup):,} rows)")



main()


usage: ipykernel_launcher.py [-h] --in INPUTS [INPUTS ...] [--out OUT]
                             [--dedup-out DEDUP_OUT] [--diff-out DIFF_OUT]
ipykernel_launcher.py: error: the following arguments are required: --in


SystemExit: 2

c:\Users\samut\anaconda3\envs\osslab\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
